# ETH MAC RTL-to-GDS Flow - Optimized for 100MHz

This notebook implements a complete RTL-to-GDS flow using Librelane for the Ethernet MAC design.

**Target Specifications:**
- Clock Frequency: 100MHz (10ns period)
- PDK: Sky130A
- Top Module: eth_top
- Clock Port: wb_clk_i

**Flow Overview:**
1. Linting and Pre-synthesis Checks
2. Synthesis
3. Floorplanning and Placement
4. Clock Tree Synthesis
5. Routing
6. Parasitic Extraction and Timing Analysis
7. Physical Verification (DRC/LVS)
8. Final Checks

In [ ]:
from librelane.config import Config

# ═══════════════════════════════════════════════════════════════
# MINIMAL CONFIGURATION FOR ETHERNET MAC @ 100MHz
# Using only CORE Librelane parameters
# ═══════════════════════════════════════════════════════════════
Config.interactive(
    "eth_top",
    PDK="sky130A",

    # ━━━ CLOCK CONFIGURATION (Most Critical) ━━━
    CLOCK_PORT="wb_clk_i",              # WISHBONE clock (main clock domain)
    CLOCK_PERIOD=10,                     # 10ns = 100MHz target frequency

    # ━━━ TIMING OPTIMIZATION (Essential for 100MHz) ━━━
    PL_RESIZER_DESIGN_OPTIMIZATIONS=1,   # Enable design optimization
    PL_RESIZER_TIMING_OPTIMIZATIONS=1,   # Enable timing optimization
    GLB_RESIZER_TIMING_OPTIMIZATIONS=1,  # Global timing optimization
    GLB_OPTIMIZE_MIRRORING=1,            # Enable mirroring

    # ━━━ OUTPUT FORMAT ━━━
    PRIMARY_GDSII_STREAMOUT_TOOL="klayout",
    
)

print("✓ Minimal configuration for ETH MAC @ 100MHz")
print("\n📊 Key Parameters:")
print("  Clock Period: 10ns (100MHz)")
print("  Timing Optimization: ENABLED")
print("\n⚠️  Note: Using minimal Librelane parameter set")
print("  Other optimizations will use Librelane defaults")


In [ ]:
from librelane.steps import Step
from librelane.state import State
import os

# ═══════════════════════════════════════════════════════════════
# WORKSPACE SETUP
# ═══════════════════════════════════════════════════════════════

# TODO: Update this path to your ETH MAC workspace
os.chdir("/home/ash29062/librelane/secular_run")

# ━━━ VERILOG SOURCE FILES ━━━
# Based on eth_speci.pdf and eth_design_document-1.pdf
VERILOG_FILES = [
    # Core modules
    "src/eth_top.v",                 # Top-level wrapper
    
    # WISHBONE interface & control
    "src/eth_wishbone.v",            # WISHBONE master/slave + DMA
    "src/eth_registers.v",           # Configuration registers
    "src/eth_register.v",            # Single register module
    
    # Transmit path
    "src/eth_txethmac.v",            # TX MAC module
    "src/eth_txstatem.v",            # TX state machine
    "src/eth_txcounters.v",          # TX counters
    "src/eth_transmitcontrol.v",     # TX flow control
    
    # Receive path  
    "src/eth_rxethmac.v",            # RX MAC module
    "src/eth_rxstatem.v",            # RX state machine
    "src/eth_rxcounters.v",          # RX counters
    "src/eth_rxaddrcheck.v",         # Address recognition
    "src/eth_receivecontrol.v",      # RX flow control
    
    # MAC control & status
    "src/eth_maccontrol.v",          # MAC control (PAUSE frames)
    "src/eth_macstatus.v",           # Status monitoring
    "src/eth_outputcontrol.v",       # Output control
    
    # MII Management
    "src/eth_miim.v",                # MII management module
    "src/eth_shiftreg.v",            # MII shift register
    "src/eth_clockgen.v",            # MII clock generator
    
    # Utilities
    "src/eth_crc.v",                 # CRC generator/checker
    "src/eth_random.v",              # Random backoff generator
    "src/eth_fifo.v",                # FIFO implementation
    "src/eth_cop.v",                 # Compare operation
    
    # Memory
    "src/eth_spram_256x32.v",        # Single-port RAM (BDs)
    
    # Configuration
    "src/ethmac_defines.v",          # Design parameters
    "src/timescale.v",               # Timescale definition
]

print("✓ Workspace configured")
print(f"✓ {len(VERILOG_FILES)} Verilog source files listed")
print("\n⚠️  Critical Design Info:")
print("  • 3 Clock Domains: wb_clk_i (100MHz), mtx_clk (25MHz), mrx_clk (25MHz)")
print("  • Internal RAM: 1KB for 128 buffer descriptors")
print("  • TX/RX FIFOs: 16 words × 32-bit each")
print("  • Estimated area: ~100k gates (0.5mm × 0.5mm @ 40% util)")


## Phase 1: Pre-Synthesis Linting and Checks
Verify RTL quality before synthesis

In [ ]:
# Verilator.Lint
# Performs static linting of Verilog/SystemVerilog code using Verilator
Lint = Step.factory.get("Verilator.Lint")

state_in = State() if 'verilator_lint' == 'verilator_lint' else lint_warnings.state_out

verilator_lint = Lint(
    VERILOG_FILES=VERILOG_FILES,
    TOP_MODULE="eth_top",
    state_in=state_in
)
verilator_lint.start()
display(verilator_lint)

In [ ]:
# Checker.LintTimingConstructs
# Checks for problematic timing constructs in RTL
LintTimingConstructs = Step.factory.get("Checker.LintTimingConstructs")

state_in = State() if 'lint_timing' == 'verilator_lint' else verilator_lint.state_out

lint_timing = LintTimingConstructs(
    VERILOG_FILES=VERILOG_FILES,
    TOP_MODULE="eth_top",
    state_in=state_in
)
lint_timing.start()
display(lint_timing)

In [ ]:
# Checker.LintErrors
# Validates absence of critical linting errors
LintErrors = Step.factory.get("Checker.LintErrors")

state_in = State() if 'lint_errors' == 'verilator_lint' else lint_timing.state_out

lint_errors = LintErrors(
    VERILOG_FILES=VERILOG_FILES,
    TOP_MODULE="eth_top",
    state_in=state_in
)
lint_errors.start()
display(lint_errors)

In [ ]:
# Checker.LintWarnings
# Checks linting warnings
LintWarnings = Step.factory.get("Checker.LintWarnings")

state_in = State() if 'lint_warnings' == 'verilator_lint' else lint_errors.state_out

lint_warnings = LintWarnings(
    VERILOG_FILES=VERILOG_FILES,
    TOP_MODULE="eth_top",
    state_in=state_in
)
lint_warnings.start()
display(lint_warnings)

## Phase 2: Synthesis
Convert RTL to gate-level netlist

In [ ]:
# Yosys.JsonHeader
# Generates JSON header for synthesis metadata

# List all Verilog source files
VERILOG_FILES = [
    "src/eth_top.v",
    "src/eth_wishbone.v",
    "src/eth_registers.v",
    "src/eth_register.v",
    "src/eth_txethmac.v",
    "src/eth_txstatem.v",
    "src/eth_txcounters.v",
    "src/eth_transmitcontrol.v",
    "src/eth_rxethmac.v",
    "src/eth_rxstatem.v",
    "src/eth_rxcounters.v",
    "src/eth_rxaddrcheck.v",
    "src/eth_receivecontrol.v",
    "src/eth_maccontrol.v",
    "src/eth_macstatus.v",
    "src/eth_outputcontrol.v",
    "src/eth_miim.v",
    "src/eth_shiftreg.v",
    "src/eth_clockgen.v",
    "src/eth_crc.v",
    "src/eth_random.v",
    "src/eth_fifo.v",
    "src/eth_cop.v",
    "src/eth_spram_256x32.v",
    "src/ethmac_defines.v",
    "src/timescale.v",
]

print(f"✓ VERILOG_FILES set: {len(VERILOG_FILES)} files")

# Run JsonHeader step with VERILOG_FILES
JsonHeader = Step.factory.get("Yosys.JsonHeader")
json_header = JsonHeader(
    state_in=lint_warnings.state_out,
    VERILOG_FILES=VERILOG_FILES,
    SYNTH_STRATEGY = "DELAY 4"
)
json_header.start()
display(json_header)


In [ ]:
# Yosys.Synthesis
# Performs logic synthesis using Yosys
Synthesis = Step.factory.get("Yosys.Synthesis")

synthesis = Synthesis(
    VERILOG_FILES=VERILOG_FILES,
    TOP_MODULE="eth_top",
    SYNTH_STRATEGY="DELAY 4",  # More aggressive optimization for 100MHz
    state_in=json_header.state_out if 1 > 0 else lint_warnings.state_out
)
synthesis.start()
display(synthesis)

In [ ]:
# Checker.YosysUnmappedCells
# Checks for unmapped cells after synthesis
YosysUnmappedCells = Step.factory.get("Checker.YosysUnmappedCells")

check_unmapped = YosysUnmappedCells(
    state_in=synthesis.state_out
)
check_unmapped.start()
display(check_unmapped)

In [ ]:
# Checker.YosysSynthChecks
# Validates synthesis quality
YosysSynthChecks = Step.factory.get("Checker.YosysSynthChecks")

synth_checks = YosysSynthChecks(
    state_in=check_unmapped.state_out
)
synth_checks.start()
display(synth_checks)

In [ ]:
# Checker.NetlistAssignStatements
# Checks for unwanted assign statements in netlist
NetlistAssignStatements = Step.factory.get("Checker.NetlistAssignStatements")

check_assigns = NetlistAssignStatements(
    state_in=synth_checks.state_out
)
check_assigns.start()
display(check_assigns)

## Phase 3: Pre-PNR Checks and Floorplanning
Prepare for physical implementation

In [ ]:
# OpenROAD.CheckSDCFiles
# Validates SDC (timing constraints) files
CheckSDCFiles = Step.factory.get("OpenROAD.CheckSDCFiles")

check_sdc = CheckSDCFiles(
    state_in=check_assigns.state_out
)
check_sdc.start()
display(check_sdc)

In [ ]:
# OpenROAD.CheckMacroInstances
# Checks macro instance placements
CheckMacroInstances = Step.factory.get("OpenROAD.CheckMacroInstances")

check_macros = CheckMacroInstances(
    state_in=check_sdc.state_out
)
check_macros.start()
display(check_macros)

In [ ]:
# OpenROAD.STAPrePNR
# Static timing analysis before place-and-route
STAPrePNR = Step.factory.get("OpenROAD.STAPrePNR")

sta_pre_pnr = STAPrePNR(
    state_in=check_macros.state_out
)
sta_pre_pnr.start()
display(sta_pre_pnr)

In [ ]:
# OpenROAD.Floorplan
# Creates chip floorplan with die area and core utilization
Floorplan = Step.factory.get("OpenROAD.Floorplan")

floorplan = Floorplan(
    state_in=sta_pre_pnr.state_out
)
floorplan.start()
display(floorplan)

In [ ]:
# Odb.CheckMacroAntennaProperties
# Verifies antenna properties of macros
CheckMacroAntennaProperties = Step.factory.get("Odb.CheckMacroAntennaProperties")

check_macro_antenna = CheckMacroAntennaProperties(
    state_in=floorplan.state_out
)
check_macro_antenna.start()
display(check_macro_antenna)

In [ ]:
# Odb.SetPowerConnections
# Sets up power and ground connections
SetPowerConnections = Step.factory.get("Odb.SetPowerConnections")

set_power = SetPowerConnections(
    state_in=check_macro_antenna.state_out
)
set_power.start()
display(set_power)

In [ ]:
# Odb.ManualMacroPlacement
# Manual placement of hard macros if needed
ManualMacroPlacement = Step.factory.get("Odb.ManualMacroPlacement")

manual_macro = ManualMacroPlacement(
    state_in=set_power.state_out
)
manual_macro.start()
display(manual_macro)

In [ ]:
# OpenROAD.CutRows
# Cuts placement rows around macros
CutRows = Step.factory.get("OpenROAD.CutRows")

cut_rows = CutRows(
    state_in=manual_macro.state_out
)
cut_rows.start()
display(cut_rows)

In [ ]:
# OpenROAD.TapEndcapInsertion
# Inserts tap cells and endcaps
TapEndcapInsertion = Step.factory.get("OpenROAD.TapEndcapInsertion")

tap_endcap = TapEndcapInsertion(
    state_in=cut_rows.state_out
)
tap_endcap.start()
display(tap_endcap)

## Phase 4: Power Distribution Network (PDN)
Create power grid for the design

In [ ]:
# Odb.AddPDNObstructions
# Adds obstructions for PDN generation
AddPDNObstructions = Step.factory.get("Odb.AddPDNObstructions")

add_pdn_obst = AddPDNObstructions(
    state_in=tap_endcap.state_out
)
add_pdn_obst.start()
display(add_pdn_obst)

In [ ]:
# OpenROAD.GeneratePDN
# Generates power distribution network - Optimized for 100MHz
GeneratePDN = Step.factory.get("OpenROAD.GeneratePDN")

generate_pdn = GeneratePDN(
    state_in=add_pdn_obst.state_out,
    # FP_PDN_VWIDTH=2,  # Wider stripes for better power delivery
    # FP_PDN_HWIDTH=2,
    # FP_PDN_VPITCH=15,  # Tighter pitch for lower IR drop
    # FP_PDN_HPITCH=15,
)
generate_pdn.start()
display(generate_pdn)

In [ ]:
# Odb.RemovePDNObstructions
# Removes PDN obstructions
RemovePDNObstructions = Step.factory.get("Odb.RemovePDNObstructions")

remove_pdn_obst = RemovePDNObstructions(
    state_in=generate_pdn.state_out
)
remove_pdn_obst.start()
display(remove_pdn_obst)

In [ ]:
# Odb.AddRoutingObstructions
# Adds routing obstructions
AddRoutingObstructions = Step.factory.get("Odb.AddRoutingObstructions")

add_route_obst = AddRoutingObstructions(
    state_in=remove_pdn_obst.state_out
)
add_route_obst.start()
display(add_route_obst)

## Phase 5: Placement
Place standard cells and perform I/O placement

In [ ]:
# OpenROAD.GlobalPlacementSkipIO
# Global placement without I/O
GlobalPlacementSkipIO = Step.factory.get("OpenROAD.GlobalPlacementSkipIO")

gpl_skip_io = GlobalPlacementSkipIO(
    state_in=add_route_obst.state_out
)
gpl_skip_io.start()
display(gpl_skip_io)

In [ ]:
# OpenROAD.IOPlacement
# Automatic I/O pin placement
IOPlacement = Step.factory.get("OpenROAD.IOPlacement")

io_placement = IOPlacement(
    state_in=gpl_skip_io.state_out
)
io_placement.start()
display(io_placement)

In [ ]:
# Odb.CustomIOPlacement
# Custom I/O placement if needed
CustomIOPlacement = Step.factory.get("Odb.CustomIOPlacement")

custom_io = CustomIOPlacement(
    state_in=io_placement.state_out
)
custom_io.start()
display(custom_io)

In [ ]:
# Odb.ApplyDEFTemplate
# Applies DEF template if provided
ApplyDEFTemplate = Step.factory.get("Odb.ApplyDEFTemplate")

apply_def = ApplyDEFTemplate(
    state_in=custom_io.state_out
)
apply_def.start()
display(apply_def)

In [ ]:
# OpenROAD.GlobalPlacement
# Global placement of standard cells
GlobalPlacement = Step.factory.get("OpenROAD.GlobalPlacement")

global_place = GlobalPlacement(
    state_in=apply_def.state_out
)
global_place.start()
display(global_place)

In [ ]:
# Odb.WriteVerilogHeader
# Writes Verilog header
WriteVerilogHeader = Step.factory.get("Odb.WriteVerilogHeader")

write_vlog_hdr = WriteVerilogHeader(
    state_in=global_place.state_out
)
write_vlog_hdr.start()
display(write_vlog_hdr)

In [ ]:
# Checker.PowerGridViolations
# Checks for power grid violations
PowerGridViolations = Step.factory.get("Checker.PowerGridViolations")

check_power_grid = PowerGridViolations(
    state_in=write_vlog_hdr.state_out
)
check_power_grid.start()
display(check_power_grid)

In [ ]:
# OpenROAD.STAMidPNR
# Timing analysis after global placement
STAMidPNR = Step.factory.get("OpenROAD.STAMidPNR")

sta_mid_pnr_1 = STAMidPNR(
    state_in=check_power_grid.state_out
)
sta_mid_pnr_1.start()
display(sta_mid_pnr_1)

In [ ]:
# OpenROAD.RepairDesignPostGPL
# Repairs design issues after global placement
RepairDesignPostGPL = Step.factory.get("OpenROAD.RepairDesignPostGPL")

repair_gpl = RepairDesignPostGPL(
    state_in=sta_mid_pnr_1.state_out
)
repair_gpl.start()
display(repair_gpl)

In [ ]:
# Odb.ManualGlobalPlacement
# Manual placement adjustments
ManualGlobalPlacement = Step.factory.get("Odb.ManualGlobalPlacement")

manual_gpl = ManualGlobalPlacement(
    state_in=repair_gpl.state_out
)
manual_gpl.start()
display(manual_gpl)

In [ ]:
# OpenROAD.DetailedPlacement
# Detailed placement of cells
DetailedPlacement = Step.factory.get("OpenROAD.DetailedPlacement")

detail_place = DetailedPlacement(
    state_in=manual_gpl.state_out
)
detail_place.start()
display(detail_place)

## Phase 6: Clock Tree Synthesis (CTS)
Build clock distribution network - Critical for 100MHz operation

In [ ]:
# OpenROAD.CTS
# Clock tree synthesis
CTS_cls = Step.factory.get("OpenROAD.CTS")

cts = CTS_cls(
    state_in=detail_place.state_out,
    CTS_BALANCE_LEVELS = "True",
    CTS_SINK_CLUSTERING_SIZE = 25,
    CTS_APPLY_NDR = "full"
)
cts.start()
display(cts)

In [ ]:
# OpenROAD.STAMidPNR
# Timing analysis after CTS
STAMidPNR_cls = Step.factory.get("OpenROAD.STAMidPNR")

sta_mid_pnr_2 = STAMidPNR_cls(
    state_in=cts.state_out
)
sta_mid_pnr_2.start()
display(sta_mid_pnr_2)

In [ ]:
# OpenROAD.ResizerTimingPostCTS
# Optimize timing after CTS
ResizerTimingPostCTS_cls = Step.factory.get("OpenROAD.ResizerTimingPostCTS")

resize_post_cts = ResizerTimingPostCTS_cls(
    state_in=sta_mid_pnr_2.state_out
)
resize_post_cts.start()
display(resize_post_cts)

In [ ]:
# OpenROAD.STAMidPNR
# Timing analysis after resizing
STAMidPNR_cls = Step.factory.get("OpenROAD.STAMidPNR")

sta_mid_pnr_3 = STAMidPNR_cls(
    state_in=resize_post_cts.state_out
)
sta_mid_pnr_3.start()
display(sta_mid_pnr_3)

## Phase 7: Global and Detailed Routing
Route all nets with antenna prevention

In [ ]:
# OpenROAD.GlobalRouting
# Global routing
GlobalRouting_cls = Step.factory.get("OpenROAD.GlobalRouting")

global_route = GlobalRouting_cls(
    state_in=sta_mid_pnr_3.state_out
)
global_route.start()
display(global_route)

In [ ]:
# OpenROAD.CheckAntennas
# Check for antenna violations
CheckAntennas_cls = Step.factory.get("OpenROAD.CheckAntennas")

check_antenna_1 = CheckAntennas_cls(
    state_in=global_route.state_out
)
check_antenna_1.start()
display(check_antenna_1)

In [ ]:
# OpenROAD.RepairDesignPostGRT
# Repair design after global routing
RepairDesignPostGRT_cls = Step.factory.get("OpenROAD.RepairDesignPostGRT")

repair_grt = RepairDesignPostGRT_cls(
    state_in=check_antenna_1.state_out
)
repair_grt.start()
display(repair_grt)

In [ ]:
# Odb.DiodesOnPorts
# Insert diodes on ports for antenna protection
DiodesOnPorts_cls = Step.factory.get("Odb.DiodesOnPorts")

diodes_ports = DiodesOnPorts_cls(
    state_in=repair_grt.state_out
)
diodes_ports.start()
display(diodes_ports)

In [ ]:
# Odb.HeuristicDiodeInsertion
# Heuristic diode insertion
HeuristicDiodeInsertion_cls = Step.factory.get("Odb.HeuristicDiodeInsertion")

heuristic_diodes = HeuristicDiodeInsertion_cls(
    state_in=diodes_ports.state_out
)
heuristic_diodes.start()
display(heuristic_diodes)

In [ ]:
# OpenROAD.RepairAntennas
# Repair antenna violations
RepairAntennas_cls = Step.factory.get("OpenROAD.RepairAntennas")

repair_antenna = RepairAntennas_cls(
    state_in=heuristic_diodes.state_out
)
repair_antenna.start()
display(repair_antenna)

In [ ]:
# OpenROAD.ResizerTimingPostGRT
# Timing optimization after global routing
ResizerTimingPostGRT_cls = Step.factory.get("OpenROAD.ResizerTimingPostGRT")

resize_post_grt = ResizerTimingPostGRT_cls(
    state_in=repair_antenna.state_out
)
resize_post_grt.start()
display(resize_post_grt)

In [ ]:
# OpenROAD.STAMidPNR
# Timing analysis post global routing
STAMidPNR_cls = Step.factory.get("OpenROAD.STAMidPNR")

sta_mid_pnr_4 = STAMidPNR_cls(
    state_in=resize_post_grt.state_out
)
sta_mid_pnr_4.start()
display(sta_mid_pnr_4)

In [ ]:
# OpenROAD.DetailedRouting
# Detailed routing
DetailedRouting_cls = Step.factory.get("OpenROAD.DetailedRouting")

detail_route = DetailedRouting_cls(
    state_in=sta_mid_pnr_4.state_out
)
detail_route.start()
display(detail_route)

In [ ]:
# Odb.RemoveRoutingObstructions
# Remove routing obstructions
RemoveRoutingObstructions_cls = Step.factory.get("Odb.RemoveRoutingObstructions")

remove_route_obst = RemoveRoutingObstructions_cls(
    state_in=detail_route.state_out
)
remove_route_obst.start()
display(remove_route_obst)

In [ ]:
# OpenROAD.CheckAntennas
# Final antenna check
CheckAntennas_cls = Step.factory.get("OpenROAD.CheckAntennas")

check_antenna_2 = CheckAntennas_cls(
    state_in=remove_route_obst.state_out
)
check_antenna_2.start()
display(check_antenna_2)

## Phase 8: Post-Routing Verification
Verify routing quality and connectivity

In [ ]:
# Checker.TrDRC
# Checks for design rule violations
TrDRC = Step.factory.get("Checker.TrDRC")

check_trdrc = TrDRC(
    state_in=check_antenna_2.state_out
)
check_trdrc.start()
display(check_trdrc)

In [ ]:
# Odb.ReportDisconnectedPins
# Reports disconnected pins
ReportDisconnectedPins = Step.factory.get("Odb.ReportDisconnectedPins")

report_disconn = ReportDisconnectedPins(
    state_in=check_trdrc.state_out
)
report_disconn.start()
display(report_disconn)

In [ ]:
# Checker.DisconnectedPins
# Checks for disconnected pins
DisconnectedPins = Step.factory.get("Checker.DisconnectedPins")

check_disconn = DisconnectedPins(
    state_in=report_disconn.state_out
)
check_disconn.start()
display(check_disconn)

In [ ]:
# Odb.ReportWireLength
# Reports total wire length
ReportWireLength = Step.factory.get("Odb.ReportWireLength")

report_wire = ReportWireLength(
    state_in=check_disconn.state_out
)
report_wire.start()
display(report_wire)

In [ ]:
# Checker.WireLength
# Checks wire length
WireLength = Step.factory.get("Checker.WireLength")

check_wire = WireLength(
    state_in=report_wire.state_out
)
check_wire.start()
display(check_wire)

## Phase 9: Fill Insertion and Parasitic Extraction
Add metal fill and extract parasitics for accurate timing

In [ ]:
# OpenROAD.FillInsertion
# Insert metal fill for density rules
FillInsertion = Step.factory.get("OpenROAD.FillInsertion")

fill_insert = FillInsertion(
    state_in=check_wire.state_out
)
fill_insert.start()
display(fill_insert)

In [ ]:
# Odb.CellFrequencyTables
# Generate cell frequency statistics
CellFrequencyTables = Step.factory.get("Odb.CellFrequencyTables")

cell_freq = CellFrequencyTables(
    state_in=fill_insert.state_out
)
cell_freq.start()
display(cell_freq)

In [ ]:
# OpenROAD.RCX
# Parasitic extraction (RC extraction)
RCX = Step.factory.get("OpenROAD.RCX")

rcx = RCX(
    state_in=cell_freq.state_out
)
rcx.start()
display(rcx)

In [ ]:
# OpenROAD.STAPostPNR
# Post-PNR static timing analysis with parasitics
STAPostPNR = Step.factory.get("OpenROAD.STAPostPNR")

sta_post_pnr = STAPostPNR(
    state_in=rcx.state_out
)
sta_post_pnr.start()
display(sta_post_pnr)

In [ ]:
# OpenROAD.IRDropReport
# IR drop analysis for power integrity
IRDropReport = Step.factory.get("OpenROAD.IRDropReport")

ir_drop = IRDropReport(
    state_in=sta_post_pnr.state_out
)
ir_drop.start()
display(ir_drop)

## Phase 10: GDSII Stream Out
Generate final layout files

In [ ]:
# Magic.StreamOut
# Generate GDSII using Magic
StreamOut = Step.factory.get("Magic.StreamOut")

magic_gds = StreamOut(
    state_in=ir_drop.state_out
)
magic_gds.start()
display(magic_gds)

In [ ]:
# KLayout.StreamOut
# Generate GDSII using KLayout
StreamOut = Step.factory.get("KLayout.StreamOut")

klayout_gds = StreamOut(
    state_in=magic_gds.state_out
)
klayout_gds.start()
display(klayout_gds)

In [ ]:
# Magic.WriteLEF
# Generate LEF file
WriteLEF = Step.factory.get("Magic.WriteLEF")

write_lef = WriteLEF(
    state_in=klayout_gds.state_out
)
write_lef.start()
display(write_lef)

In [ ]:
# Odb.CheckDesignAntennaProperties
# Final antenna property check
CheckDesignAntennaProperties = Step.factory.get("Odb.CheckDesignAntennaProperties")

check_design_antenna = CheckDesignAntennaProperties(
    state_in=write_lef.state_out
)
check_design_antenna.start()
display(check_design_antenna)

## Phase 11: XOR Verification
Compare different GDS outputs

In [ ]:
# KLayout.XOR
# Perform XOR comparison
XOR_cls = Step.factory.get("KLayout.XOR")

klayout_xor = XOR_cls(
    state_in=check_design_antenna.state_out
)
klayout_xor.start()
display(klayout_xor)

In [ ]:
# Checker.XOR
# Check XOR results
XOR_cls = Step.factory.get("Checker.XOR")

check_xor = XOR_cls(
    state_in=klayout_xor.state_out
)
check_xor.start()
display(check_xor)

## Phase 12: Design Rule Check (DRC)
Verify layout meets foundry rules

In [ ]:
# Magic.DRC
# DRC check using Magic
DRC = Step.factory.get("Magic.DRC")

magic_drc = DRC(
    state_in=check_xor.state_out
)
magic_drc.start()
display(magic_drc)

In [ ]:
import dill
dill.dump_session('notebook_env.db')


## Phase 13: LVS and Equivalence Checking
Verify layout matches schematic

In [ ]:
# Magic.SpiceExtraction
# Extract SPICE netlist from layout
SpiceExtraction = Step.factory.get("Magic.SpiceExtraction")

spice_extract = SpiceExtraction(
    state_in=magic_drc.state_out
)
spice_extract.start()
display(spice_extract)

In [ ]:
# Checker.IllegalOverlap
# Check for illegal overlaps
IllegalOverlap = Step.factory.get("Checker.IllegalOverlap")

check_overlap = IllegalOverlap(
    state_in=spice_extract.state_out
)
check_overlap.start()
display(check_overlap)

In [ ]:
# Netgen.LVS
# Layout vs Schematic verification
LVS = Step.factory.get("Netgen.LVS")

lvs = LVS(
    state_in=check_overlap.state_out
)
lvs.start()
display(lvs)

In [ ]:
# Checker.LVS
# Validate LVS results
LVS = Step.factory.get("Checker.LVS")

check_lvs = LVS(
    state_in=lvs.state_out
)
check_lvs.start()
display(check_lvs)

## Phase 14: Final Timing Verification
Validate 100MHz timing requirements are met

In [ ]:
# Checker.SetupViolations
# Check for setup time violations
SetupViolations = Step.factory.get("Checker.SetupViolations")

check_setup = SetupViolations(
    state_in=check_lvs.state_out
)
check_setup.start()
display(check_setup)

In [ ]:
# Checker.HoldViolations
# Check for hold time violations
HoldViolations = Step.factory.get("Checker.HoldViolations")

check_hold = HoldViolations(
    state_in=check_setup.state_out
)
check_hold.start()
display(check_hold)

In [ ]:
# Checker.MaxSlewViolations
# Check for maximum slew violations
MaxSlewViolations = Step.factory.get("Checker.MaxSlewViolations")

check_slew = MaxSlewViolations(
    state_in=check_hold.state_out
)
check_slew.start()
display(check_slew)

In [ ]:
# Checker.MaxCapViolations
# Check for maximum capacitance violations
MaxCapViolations = Step.factory.get("Checker.MaxCapViolations")

check_cap = MaxCapViolations(
    state_in=check_slew.state_out
)
check_cap.start()
display(check_cap)

## Phase 15: Final Manufacturability Report
Generate comprehensive manufacturing metrics

In [ ]:
# Misc.ReportManufacturability
# Generates final manufacturability report
ReportManufacturability = Step.factory.get("Misc.ReportManufacturability")

report_mfg = ReportManufacturability(
    state_in=check_cap.state_out
)
report_mfg.start()
display(report_mfg)

In [ ]:
# 🔧 Final metrics summary table for eth_top
# Run this AFTER your last step (e.g. step21_top) has finished.

import pandas as pd
from decimal import Decimal

# CHANGE THIS if your final step variable has a different name:
final_state = sta_post_pnr.state_out
metrics = final_state.metrics

def to_float(val):
    """Safely convert Decimal/str/None/etc. to float or None."""
    if val is None:
        return None
    if isinstance(val, Decimal):
        return float(val)
    try:
        return float(val)
    except Exception:
        return None

def pick_metric(*names, default=None):
    """Try several metric keys, return first that exists."""
    for n in names:
        if n in metrics:
            return metrics[n]
    return default

# -------- Timing / clock --------
target_period = to_float(pick_metric("CLOCK_PERIOD", "config__CLOCK_PERIOD", default=15.0))
setup_slack  = to_float(pick_metric("timing__setup__ws",  "signoff__timing__setup__ws",  default=0.0))
hold_slack   = to_float(pick_metric("timing__hold__ws",   "signoff__timing__hold__ws",   default=0.0))

target_freq = 1000.0 / target_period if target_period else None
if setup_slack is not None:
    if setup_slack >= 0:
        achieved_period = target_period - setup_slack
    else:
        achieved_period = target_period + abs(setup_slack)
    achieved_freq = 1000.0 / achieved_period
else:
    achieved_period = None
    achieved_freq = None

# -------- Area / utilization --------
die_area_um2  = to_float(pick_metric("design__die__area",  "floorplan__design__die__area",  default=0.0))
core_area_um2 = to_float(pick_metric("design__core__area", "floorplan__design__core__area", default=0.0))
util_percent  = to_float(pick_metric("design__instance__utilization",
                                     "floorplan__design__instance__utilization",
                                     default=0.0)) * 100.0

die_area_mm2  = die_area_um2 / 1e6 if die_area_um2 else 0.0
core_area_mm2 = core_area_um2 / 1e6 if core_area_um2 else 0.0

die_w_um = to_float(pick_metric("design__die__width__um",  "floorplan__design__die__width__um",  default=0.0))
die_h_um = to_float(pick_metric("design__die__height__um", "floorplan__design__die__height__um", default=0.0))

# -------- Cell counts --------
total_cells = to_float(pick_metric("design__instance__count",
                                   "signoff__design__instance__count",
                                   "floorplan__design__instance__count",
                                   default=0.0))
seq_cells   = to_float(pick_metric("design__instance__count__sequential",
                                   "signoff__design__instance__count__sequential",
                                   default=0.0))
buf_cells   = to_float(pick_metric("design__instance__count__buffer",
                                   "signoff__design__instance__count__buffer",
                                   default=0.0))
comb_cells  = (total_cells or 0.0) - (seq_cells or 0.0) - (buf_cells or 0.0)

# -------- I/O count --------
total_ios = to_float(pick_metric("design__io__count",
                                 "signoff__design__io",
                                 "floorplan__design__io",
                                 default=0.0))

# -------- Routing metrics --------
wire_len_um = to_float(pick_metric("route__wirelength__estimated",
                                   "route__wire_length",
                                   "signoff__route__wirelength__estimated",
                                   default=0.0))
vias = to_float(pick_metric("route__vias",
                            "route__via_count",
                            "signoff__route__vias",
                            default=0.0))

# -------- Power (prefer real metrics; else rough estimate) --------
# Try to pick a total power metric if Librelane populated it.
pwr_keys = [k for k in metrics.keys() if k.endswith("power__total") or k.endswith("power__total__mw")]
pwr_total_mw = None
if pwr_keys:
    # take the last key (usually the latest stage, e.g. signoff__power__total)
    pwr_total_mw = to_float(metrics[pwr_keys[-1]]) * 1e3 if "total" in pwr_keys[-1] else to_float(metrics[pwr_keys[-1]])

# Fallback rough estimate if no explicit power metric
if pwr_total_mw is None:
    # very rough: assume ~1 mW / MHz per 1e4 gates on Sky130, scale by cell count and freq
    if achieved_freq and total_cells:
        pwr_total_mw = (total_cells / 1e4) * (achieved_freq / 100.0)

# -------- Build table --------
rows = [
    ("Clock target (MHz)",      f"{target_freq:.2f}"        if target_freq else "N/A"),
    ("Clock achieved (MHz)",    f"{achieved_freq:.2f}"      if achieved_freq else "N/A"),
    ("Clock period target (ns)",f"{target_period:.3f}"      if target_period else "N/A"),
    ("Setup slack (ns)",        f"{setup_slack:+.3f}"       if setup_slack is not None else "N/A"),
    ("Hold slack (ns)",         f"{hold_slack:+.3f}"        if hold_slack is not None else "N/A"),

    ("Die area (mm²)",          f"{die_area_mm2:.4f}"       if die_area_mm2 else "N/A"),
    ("Core area (mm²)",         f"{core_area_mm2:.4f}"      if core_area_mm2 else "N/A"),
    ("Die size (µm)",           f"{die_w_um:.0f} × {die_h_um:.0f}" if die_w_um and die_h_um else "N/A"),
    ("Core utilization (%)",    f"{util_percent:.1f}"       if util_percent else "N/A"),

    ("Total std cells",         f"{int(total_cells):,}"     if total_cells else "N/A"),
    ("Flip‑flops",              f"{int(seq_cells):,}"       if seq_cells else "N/A"),
    ("Buffers / inverters",     f"{int(buf_cells):,}"       if buf_cells else "N/A"),
    ("Combinational cells",     f"{int(comb_cells):,}"      if comb_cells else "N/A"),

    ("Total I/O pins",          f"{int(total_ios):,}"       if total_ios else "N/A"),

    ("Total wire length (mm)",  f"{wire_len_um/1000:.2f}"   if wire_len_um else "N/A"),
    ("Total vias",              f"{int(vias):,}"            if vias else "N/A"),

    ("Total power est. (mW)",   f"{pwr_total_mw:.1f}"       if pwr_total_mw else "N/A"),
]

df = pd.DataFrame(rows, columns=["Metric", "Value"])
display(df.style.hide(axis="index"))


In [ ]:
final_state = report_mfg.state_out
for k in sorted(final_state.metrics.keys()):
    print(k, "=>", final_state.metrics[k])


In [ ]:
from tabulate import tabulate
import pandas as pd

# Get final state metrics
final_state = report_mfg.state_out
metrics = final_state.metrics

# Create table data
table_data = []
for key in sorted(metrics.keys()):
    value = metrics[key]
    # Safely convert Decimal/None to display-friendly format
    if value is None:
        display_value = "None"
    elif hasattr(value, 'quantize'):  # Decimal
        display_value = f"{float(value):.4f}"
    else:
        display_value = str(value)
    
    table_data.append([key, display_value])

# Print beautiful table
print("="*80)
print("FINAL MANUFACTURABILITY METRICS REPORT")
print("="*80)
print(tabulate(table_data, headers=["Metric", "Value"], tablefmt="grid", maxcolwidths=[50, 20]))

# Also create a pandas DataFrame for easy filtering/export
df_metrics = pd.DataFrame(table_data, columns=["Metric", "Value"])
print("\n" + "="*80)
print("PANDAS DATAFRAME CREATED (df_metrics)")
print("• Filter: df_metrics[df_metrics['Metric'].str.contains('area')]")
print("• Export: df_metrics.to_csv('final_metrics.csv')")
print("• Sort: df_metrics.sort_values('Metric')")


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
from tabulate import tabulate

# Get final state metrics
final_state = report_mfg.state_out
metrics = final_state.metrics

# Create DataFrame
df = pd.DataFrame([
    {'Metric': key, 'Value': metrics[key]} 
    for key in sorted(metrics.keys())
])

# Safely format values for display
def format_value(val):
    if val is None:
        return "None"
    elif hasattr(val, 'quantize'):  # Decimal
        return f"{float(val):.4f}"
    else:
        return str(val)

df['Value'] = df['Value'].apply(format_value)

# Generate Markdown table
markdown_table = df.to_markdown(index=False, tablefmt="pipe", headers=["Metric", "Value"])

print("## 🎯 Final Manufacturability Metrics")
print("```")
print(markdown_table)
print("```")

# Display as interactive DataFrame too
display(df)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
from tabulate import tabulate

def generate_summary(final_state):
    """Generate complete manufacturability summary from report_mfg.state_out"""
    
    metrics = final_state.metrics
    
    def safe_float(val):
        if val is None: return 0.0
        try: return float(val)
        except: return 0.0
    
    def safe_int(val):
        return int(safe_float(val))
    
    def format_value(val):
        if val is None: return "None"
        elif hasattr(val, 'quantize'):  # Decimal
            return f"{float(val):.4f}"
        else:
            return str(val)
    
    # Extract key metrics
    core_area = safe_float(metrics.get('design__core__area', 0))
    die_area = safe_float(metrics.get('design__die__area', 0))
    inst_area = safe_float(metrics.get('design__instance__area', 0))
    inst_count = safe_int(metrics.get('design__instance__count', 0))
    stdcell_count = safe_int(metrics.get('design__instance__count__stdcell', 0))
    utilization = safe_float(metrics.get('design__instance__utilization', 0)) * 100
    
    total_power = safe_float(metrics.get('power__total', 0))
    internal_power = safe_float(metrics.get('power__internal__total', 0))
    switching_power = safe_float(metrics.get('power__switching__total', 0))
    leakage_power = safe_float(metrics.get('power__leakage__total', 0))
    
    nets = safe_int(metrics.get('route__net', 0))
    wirelength = safe_float(metrics.get('route__wirelength', 0))
    vias = safe_int(metrics.get('route__vias', 0))
    
    setup_wns = safe_float(metrics.get('timing__setup__wns', 0))
    setup_tns = safe_float(metrics.get('timing__setup__tns', 0))
    hold_wns = safe_float(metrics.get('timing__hold__wns', 0))
    hold_tns = safe_float(metrics.get('timing__hold__tns', 0))
    
    clk_skew_setup = safe_float(metrics.get('clock__skew__worst_setup', 0))
    clk_skew_hold = safe_float(metrics.get('clock__skew__worst_hold', 0))
    
    antenna_nets = safe_int(metrics.get('antenna__violating__nets', 0))
    max_cap_viol = safe_int(metrics.get('design__max_cap_violation__count', 0))
    max_fanout_viol = safe_int(metrics.get('design__max_fanout_violation__count', 0))
    max_slew_viol = safe_int(metrics.get('design__max_slew_violation__count', 0))
    
    ir_drop_worst = safe_float(metrics.get('ir__drop__worst', 0))
    lvs_errors = safe_int(metrics.get('design__lvs_error__count', 0))
    drc_errors = safe_int(metrics.get('magic__drc_error__count', 0))
    
    # Calculated metrics
    clk_period = 15.0
    fmax = 1000 / max(1, clk_period + max(0, -setup_wns))
    power_density = total_power / (core_area / 1e6) * 1e6 if core_area > 0 else 0
    area_eff = inst_area / inst_count if inst_count > 0 else 0
    wire_density = wirelength / inst_count if inst_count > 0 else 0
    
    # 1. FULL METRICS TABLE (your original code)
    df_full = pd.DataFrame([
        {'Metric': key, 'Value': format_value(metrics[key])} 
        for key in sorted(metrics.keys())
    ])
    
    # 2. SUMMARY TABLE
    summary_table = [
        ["Performance", "Target Clock", f"{1000/clk_period:.1f} MHz", f"{clk_period:.1f} ns"],
        ["Performance", "Est. Fmax", f"{fmax:.1f} MHz", f"{clk_period+max(0,-(setup_wns)):.3f} ns"],
        ["Performance", "Setup WNS", f"{setup_wns:.4f} ns", ""],
        
        ["Area", "Die Area", f"{die_area/1e6:.3f} mm²", ""],
        ["Area", "Core Area", f"{core_area/1e6:.3f} mm²", ""],
        ["Area", "Instance Area", f"{inst_area/1e6:.3f} mm²", ""],
        ["Area", "Utilization", f"{utilization:.1f}%", ""],
        ["Area", "Cells", f"{inst_count:,}", f"{area_eff:.1f} μm²/cell"],
        
        ["Power", "Total Power", f"{total_power*1e6:.2f} μW", f"{power_density:.1f} μW/mm²"],
        ["Power", "Internal", f"{internal_power*1e6:.2f} μW", f"{internal_power/total_power*100:.0f}%" if total_power > 0 else ""],
        ["Power", "Switching", f"{switching_power*1e6:.2f} μW", f"{switching_power/total_power*100:.0f}%" if total_power > 0 else ""],
        ["Power", "Leakage", f"{leakage_power*1e6:.2f} μW", f"{leakage_power/total_power*100:.1f}%" if total_power > 0 else ""],
        
        ["Routing", "Nets", f"{nets:,}", ""],
        ["Routing", "Wirelength", f"{wirelength/1e6:.1f} mm", f"{wire_density:.1f} μm/cell"],
        ["Routing", "Vias", f"{vias:,}", ""],
        
        ["Timing", "Setup WNS", f"{setup_wns:.4f} ns", ""],
        ["Timing", "Setup TNS", f"{setup_tns:.1f} ns", ""],
        ["Timing", "Hold WNS", f"{hold_wns:.4f} ns", ""],
        ["Timing", "Hold TNS", f"{hold_tns:.1f} ns", ""],
        ["Timing", "Clk Skew Setup", f"{clk_skew_setup:.4f} ns", ""],
        ["Timing", "Clk Skew Hold", f"{clk_skew_hold:.4f} ns", ""],
        
        ["Quality", "Antenna Nets", f"{antenna_nets}", ""],
        ["Quality", "Max Cap Violations", f"{max_cap_viol}", ""],
        ["Quality", "Max Fanout Violations", f"{max_fanout_viol}", ""],
        ["Quality", "Max Slew Violations", f"{max_slew_viol}", ""],
        ["Quality", "LVS Errors", f"{lvs_errors}", ""],
        ["Quality", "DRC Errors", f"{drc_errors}", ""],
        ["Quality", "IR Drop Worst", f"{ir_drop_worst*100:.4f}%", ""]
    ]
    
    # Print results
    print("## Final Manufacturability Summary")
    print(tabulate(summary_table, headers=["Category", "Metric", "Value", "Details"], 
                   tablefmt="grid", colalign=["left", "left", "right", "left"]))
    
    print("\n## Complete Metrics Table")
    print(df_full.to_markdown(index=False, tablefmt="pipe"))
    
    # Interactive displays
    print("\n## Interactive DataFrames")
    display(Markdown("### Summary Table"))
    display(pd.DataFrame(summary_table[1:], columns=summary_table[0]))
    
    display(Markdown("### Full Metrics"))
    display(df_full.head(50))  # First 50 for readability
    
    # Save files
    pd.DataFrame(summary_table[1:], columns=summary_table[0]).to_csv('summary_table.csv', index=False)
    df_full.to_csv('full_metrics.csv', index=False)
    
    print("\nFiles saved:")
    print("- summary_table.csv")
    print("- full_metrics.csv")
    
    return df_full, pd.DataFrame(summary_table[1:], columns=summary_table[0])

# Generate complete summary
df_full, df_summary = generate_summary(report_mfg.state_out)


In [ ]:
import dill
dill.dump_session('notebook_env.db')
